**Já usamos as métricas (MAE,RMSE e R²) para avaliar os resultados do modelo nas fases de dados conhecidos e desconhecidos:**

80 % Dados ─────────────────────────► TREINO (aprendizado)


80% Dados ─────────────────────────► TESTE (previsão de novos preços)



A pergunta agora é:
Se utilizamos outras partes dos dados o desempenho ainda é bom?

Nosso modelo candidato foi o Random Forest, então precisamos validar esse modelo.

- Vamos segregar os dados de treinamento, ou seja, pegar os 80% dos dados\
criar uma validação cruzada entre 5 partes chamdas de fold, pelo método de cross validation do scikit-learn.


│ Fold 1 │ Fold 2 │ Fold 3 │ Fold 4 │ Fold 5 │


Na sequencia realizamos rodadas aletórias:
- uma parte serve como validador e treinamos os outros 4 e assim por diante
como no esquema:



VALIDAÇÃO
│ Fold 1 │

TREINAMENTO
│ Fold 2 │ Fold 3 │ Fold 4 │ Fold 5 │

POR FIM o desempenho médio:

Fold 1 → resultado\
Fold 2 → resultado\
Fold 3 → resultado\
Fold 4 → resultado\
Fold 5 → resultado\
         ↓\
      MÉDIA\
         ↓\
 desempenho médio

A validação vai nos permitir pensar nos aspectos de otimização do modelo

In [1]:
# Importando bibliotecas
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split #para dividir os dados em treino e teste
from sklearn.compose import ColumnTransformer # para aplicar diferentes transformações em diferentes colunas do dataset
from sklearn.pipeline import Pipeline # para criar um pipeline de pré-processamento e modelagem
from sklearn.impute import SimpleImputer # para lidar com valores ausentes
from sklearn.preprocessing import OneHotEncoder # para codificar variáveis categóricas
from sklearn.preprocessing import StandardScaler # para padronizar os dados

#recarga da base
df = pd.read_csv(r'C:\Users\marciohenrique\Documents\Projetos\github\ml_stat\bases_dados\base_imoveis.csv')

In [2]:

colunas = [
    "area_m2",
    "quartos",
    "banheiros",
    "vagas_garagem",
    "idade_anos",
    "localidade",
    "tipo"
]

val_numer = [
    "area_m2",
    "quartos",
    "banheiros",
    "vagas_garagem",
    "idade_anos"
]

val_categ = [
    "localidade",
    "tipo"
]

df = df.dropna(subset=["preco_venda"]).copy() #copia da base de dados para não alterar a original


transf_val_num= Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")), # substitui valores nulos pela mediana
    ]
)


transf_val_cat= Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")), # substitui valores nulos pela moda
    ("onehot", OneHotEncoder(handle_unknown="ignore")) # codifica variáveis categóricas
    ]
)

In [3]:
#CONFIG TREINO E TESTE
X = df[colunas]
y = df["preco_venda"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # 80% treino e 20% teste

"""Preprocessamento dos dados, aplicando as transformações definidas anteriormente nas colunas numéricas e categóricas """
preprocessamento = ColumnTransformer(
    transformers=[
        ("num", transf_val_num, val_numer),
        ("cat", transf_val_cat, val_categ)
    ]
)

In [9]:
#Import da validação cruzada KFold e do modelo Random Forest Regressor para o pipeline de validação cruzada
from sklearn.model_selection import cross_validate, KFold
from sklearn.ensemble import RandomForestRegressor

mmodelo_rf_cv = Pipeline(
    steps=[
        ("preprocessamento", preprocessamento),
        (
            "modelo",
            RandomForestRegressor(
                n_estimators=200,
                max_depth=None,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [8]:
#Configurando a validação
cv = KFold(
    n_splits=5, #dividindo os dados em 5 partes
    shuffle=True, #embaralhando os dados antes de dividir
    random_state=42 #reprodução da mesma divisão dos dados em diferentes execuções do código
)

In [10]:
resultados_cv = cross_validate(
    modelo_rf_cv,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2"
    },
    n_jobs=-1
)

In [12]:
mae_cv = -resultados_cv["test_MAE"]
rmse_cv = -resultados_cv["test_RMSE"]
r2_cv = resultados_cv["test_R2"]

resultados_folds = pd.DataFrame({
    "Fold": [1, 2, 3, 4, 5],
    "MAE": mae_cv,
    "RMSE": rmse_cv,
    "R²": r2_cv
})

display(resultados_folds)

,Fold,MAE,RMSE,R²
0,1,151671.575374,275890.826257,0.853043
1,2,154167.099372,251199.423962,0.911258
2,3,142905.827005,239801.521501,0.899853
3,4,143606.650848,250701.710186,0.854653
4,5,142166.022076,218788.056514,0.908993


In [14]:
print(f"MAE médio: R$ {mae_cv.mean():,.2f}")
print(f"RMSE médio: R$ {rmse_cv.mean():,.2f}")
print(f"R² médio: {r2_cv.mean():.4f}")

print(f"Desvio padrão MAE: R$ {mae_cv.std():,.2f}")
print(f"Desvio padrão RMSE: R$ {rmse_cv.std():,.2f}")
print(f"Desvio padrão R²: {r2_cv.std():.4f}")

MAE médio: R$ 146,903.43
RMSE médio: R$ 247,276.31
R² médio: 0.8856
Desvio padrão MAE: R$ 4,995.77
Desvio padrão RMSE: R$ 18,511.44
Desvio padrão R²: 0.0262
